# Training Experiments — SigLIP-B/16-384 + LoRA

Scratch notebook for trying training ideas quickly.

**To run a new experiment:** change the config in cell 2, then Run All.  
**B0 frozen baseline** (1.43% mIoU) is fixed — see `report.md` and `03_siglip_b0_eval.ipynb`.  
Results are logged to W&B project `region-grounded`.

Trains **M_human** only: L_global (MHAP pooler) + λ·L_region with human Flickr30k Entities bboxes.

| Cell | Purpose |
|------|---------|
| 0–2  | Config — **edit here** |
| 3–7  | Infrastructure (data, model, losses, helpers) |
| 8    | `quick_voc_eval` |
| 9    | `train_one_epoch` + optimizer/scheduler |
| 10+  | Experiment run |

In [1]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
# Run once at the start of a Colab session.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft>=0.10', 'datasets', 'transformers>=4.40', 'huggingface_hub',
    'tqdm', 'Pillow', 'wandb', 'torchao>=0.16.0',
], check=True)
print('Dependencies installed.')

Dependencies installed.


In [2]:
# ── 1. Imports ─────────────\───────────────────────────────────────────────────
import os, sys, io, random, getpass, zipfile, urllib.request, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torchvision.datasets as tvd

from transformers import SiglipModel, AutoProcessor
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from peft import get_peft_model, LoraConfig
import wandb

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU  : NVIDIA L4
VRAM : 23.7 GB


In [3]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
# ← EDIT THIS CELL to change the experiment.

CFG = dict(
    model_id       = 'google/siglip-base-patch16-384',
    eval_size      = 384,
    patch_size     = 16,

    # LoRA
    lora_rank      = 4,
    lora_alpha     = 16,
    lora_dropout   = 0.1,
    lora_targets   = ['q_proj', 'v_proj'],

    # Training
    batch_size     = 32,
    region_batch   = 64,       # phrase-bbox pairs per region loss step
    lr             = 2e-4,
    weight_decay   = 0.01,
    epochs         = 1,
    warmup_steps   = 100,
    prompt         = 'a photo of a {}',

    # Region loss weight  (>0 = global + region)
    lambda_region  = 0.5,

    # Eval
    eval_images    = 200,
    tau_seg        = 0.0,

    ckpt_dir       = 'checkpoints',
)
CFG['n_side'] = CFG['eval_size'] // CFG['patch_size']  # 24
Path(CFG['ckpt_dir']).mkdir(exist_ok=True)

# Known frozen baseline — do not recompute here; see report.md
B0_MIOU = 1.43

print(CFG)
print(f'B0 baseline (frozen, from report.md): {B0_MIOU}%')

{'model_id': 'google/siglip-base-patch16-384', 'eval_size': 384, 'patch_size': 16, 'lora_rank': 4, 'lora_alpha': 16, 'lora_dropout': 0.1, 'lora_targets': ['q_proj', 'v_proj'], 'batch_size': 32, 'region_batch': 64, 'lr': 0.0002, 'weight_decay': 0.01, 'epochs': 1, 'warmup_steps': 100, 'prompt': 'a photo of a {}', 'lambda_region': 0.5, 'eval_images': 200, 'tau_seg': 0.0, 'ckpt_dir': 'checkpoints', 'n_side': 24}
B0 baseline (frozen, from report.md): 1.43%


In [4]:
# ── 2b. W&B init ─────────────────────────────────────────────────────────────
wandb.login()

run = wandb.init(
    project = 'region-grounded',
    name    = f'mhuman_lam{CFG["lambda_region"]}_r{CFG["lora_rank"]}',
    config  = CFG,
    tags    = ['siglip', 'flickr30k', 'lora'],
)
print(f'W&B run: {run.url}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sidraj (sidraj-university-of-chicago-charter-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/sidraj-university-of-chicago-charter-school/region-grounded/runs/8b71jyl9


In [5]:
# ── 3. Data — load from Drive cache or download fresh ───────────────────────
# First run  : downloads parquet + entities, zips, uploads to Drive (~15 min).
# Later runs : mounts Drive, extracts the zip locally (~2 min).

from google.colab import drive

GDRIVE_ROOT  = Path('/content/drive')
drive.mount(str(GDRIVE_ROOT))

GDRIVE_ZIP   = GDRIVE_ROOT / 'MyDrive' / 'region-grounded-data' / 'flickr30k_full.zip'
LOCAL_DATA   = Path('data/flickr30k')
PQ_DIR       = LOCAL_DATA / 'parquet'
ENTITIES_DIR = LOCAL_DATA / 'entities'
ANN_DIR      = ENTITIES_DIR / 'Annotations'
SENT_DIR     = ENTITIES_DIR / 'Sentences'

if GDRIVE_ZIP.exists():
    # ── Fast path ─────────────────────────────────────────────────────────────
    sz_gb = GDRIVE_ZIP.stat().st_size / 1e9
    print(f'Drive cache found ({sz_gb:.1f} GB). Extracting...')
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(str(GDRIVE_ZIP)) as zf:
        zf.extractall('.')
    print('Extracted.')

else:
    # ── First run: download everything, then zip to Drive ─────────────────────
    PQ_DIR.mkdir(parents=True, exist_ok=True)
    ENTITIES_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Flickr30k parquet shards
    REPO_ID  = 'nlphuji/flickr30k'
    api      = HfApi(token=os.environ['HF_TOKEN'])
    pq_names = sorted(
        f for f in api.list_repo_files(REPO_ID, repo_type='dataset',
                                       revision='refs/convert/parquet')
        if f.endswith('.parquet')
    )
    print(f'Downloading {len(pq_names)} parquet shard(s)...')
    for fname in pq_names:
        cached = hf_hub_download(
            repo_id=REPO_ID, filename=fname, repo_type='dataset',
            revision='refs/convert/parquet', token=os.environ['HF_TOKEN'],
        )
        shutil.copy(cached, PQ_DIR / Path(fname).name)
    print(f'Parquet shards → {PQ_DIR}')

    # 2. Flickr30k Entities utils + annotations
    UTILS_URL = ('https://raw.githubusercontent.com/bryanplummer/'
                 'flickr30k_entities/master/flickr30k_entities_utils.py')
    urllib.request.urlretrieve(UTILS_URL, ENTITIES_DIR / 'flickr30k_entities_utils.py')

    ZIP_URL   = ('https://github.com/bryanplummer/flickr30k_entities/'
                 'raw/master/annotations.zip')
    TMP_ANN   = Path('/tmp/annotations.zip')
    if not ANN_DIR.is_dir():
        print('Downloading Entities annotations (~25 MB)...')
        urllib.request.urlretrieve(ZIP_URL, TMP_ANN)
        with zipfile.ZipFile(TMP_ANN) as zf:
            zf.extractall(ENTITIES_DIR)
        print('Extracted.')

    # 3. Zip all data and save to Drive (ZIP_STORED: parquet is already compressed)
    TMP_ZIP = Path('/tmp/flickr30k_full.zip')
    total   = sum(1 for f in LOCAL_DATA.rglob('*') if f.is_file())
    print(f'Zipping {total} files (no compression — parquet is pre-compressed)...')
    with zipfile.ZipFile(str(TMP_ZIP), 'w', zipfile.ZIP_STORED) as zf:
        for f in LOCAL_DATA.rglob('*'):
            if f.is_file():
                zf.write(str(f), str(f))
    sz_gb = TMP_ZIP.stat().st_size / 1e9
    GDRIVE_ZIP.parent.mkdir(parents=True, exist_ok=True)
    print(f'Uploading {sz_gb:.1f} GB to Drive...')
    shutil.copy(str(TMP_ZIP), str(GDRIVE_ZIP))
    TMP_ZIP.unlink()
    print(f'Saved to {GDRIVE_ZIP}')

# Load parquet into HuggingFace Dataset
local_pq = sorted(str(f) for f in PQ_DIR.glob('*.parquet'))
hf_data  = load_dataset('parquet', data_files={'data': local_pq})['data']
print(f'Flickr30k rows: {len(hf_data):,}')
print('Columns:', hf_data.column_names)

# Import Entities utils
if str(ENTITIES_DIR) not in sys.path:
    sys.path.insert(0, str(ENTITIES_DIR))
from flickr30k_entities_utils import get_sentence_data, get_annotations

print(f'Annotations: {len(list(ANN_DIR.iterdir())):,} XML files')
print(f'Sentences  : {len(list(SENT_DIR.iterdir())):,} txt files')


Mounted at /content/drive
Drive cache found (4.4 GB). Extracting...
Extracted.


Generating data split: 0 examples [00:00, ? examples/s]

Flickr30k rows: 31,014
Columns: ['image', 'caption', 'sentids', 'split', 'img_id', 'filename']
Annotations: 31,783 XML files
Sentences  : 31,783 txt files


In [6]:
# ── 4. Dataset ────────────────────────────────────────────────────────────────
# One item per image. Returns:
#   pil_image  : PIL image (original size, unprocessed)
#   caption    : one randomly sampled caption string
#   orig_size  : (W, H) of the original image — needed for bbox scaling
#   phrase_boxes: list of (phrase_str, [x1,y1,x2,y2]) in original pixel coords

class FlickrSigLIPDataset(Dataset):

    def __init__(self, hf_data, ann_dir: Path, sent_dir: Path, split: str = 'train'):
        self.ann_dir  = ann_dir
        self.sent_dir = sent_dir
        self.rows     = [r for r in hf_data if r['split'] == split]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        # decode PIL image
        img_field = row['image']
        if isinstance(img_field, PILImage.Image):
            pil = img_field.convert('RGB')
        else:
            pil = PILImage.open(io.BytesIO(img_field['bytes'])).convert('RGB')
        orig_size = pil.size  # (W, H)

        caption = random.choice(row['caption'])

        # phrase-bbox pairs from Entities annotations
        stem = row['filename'].replace('.jpg', '')
        try:
            anns  = get_annotations(str(self.ann_dir  / f'{stem}.xml'))
            sents = get_sentence_data(str(self.sent_dir / f'{stem}.txt'))
        except Exception:
            return pil, caption, orig_size, []

        phrase_boxes = []
        seen = set()
        for sent in sents:
            for phrase in sent['phrases']:
                pid = phrase['phrase_id']
                if pid in seen or pid not in anns['boxes']:
                    continue
                seen.add(pid)
                for box in anns['boxes'][pid]:
                    phrase_boxes.append((phrase['phrase'], box))

        return pil, caption, orig_size, phrase_boxes

    @staticmethod
    def collate_fn(batch):
        pils, captions, orig_sizes, phrase_boxes = zip(*batch)
        return list(pils), list(captions), list(orig_sizes), list(phrase_boxes)


train_ds = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='train')
val_ds   = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='val')
print(f'Train: {len(train_ds):,} images')
print(f'Val  : {len(val_ds):,} images')

# Sanity check one sample
pil0, cap0, sz0, pb0 = train_ds[0]
print(f'\nSample: size={sz0}  caption="{cap0[:60]}…"')
print(f'phrase-box pairs: {len(pb0)}  e.g. {pb0[0] if pb0 else "none"}')

Train: 29,000 images
Val  : 1,014 images

Sample: size=(333, 500)  caption="Two men in green shirts are standing in a yard.…"
phrase-box pairs: 12  e.g. ('Two young guys', [158, 124, 218, 334])


In [7]:
# ── 5. Load SigLIP + apply LoRA ───────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(CFG['model_id'],
                                          token=os.environ['HF_TOKEN'])
base_model = SiglipModel.from_pretrained(CFG['model_id'],
                                         token=os.environ['HF_TOKEN'])

lora_cfg = LoraConfig(
    r                = CFG['lora_rank'],
    lora_alpha       = CFG['lora_alpha'],
    lora_dropout     = CFG['lora_dropout'],
    target_modules   = CFG['lora_targets'],
    bias             = 'none',
)
model = get_peft_model(base_model, lora_cfg)
model = model.to(DEVICE)
model.print_trainable_parameters()

# logit_scale and logit_bias are SigLIP's contrastive temperature parameters.
# Keep them trainable — they are NOT in LoRA modules.
for n, p in model.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/814M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 203,742,722 || trainable%: 0.1447
Trainable: 294,914 / 203,742,722  (0.14%)


In [8]:
# ── 6. Loss functions ─────────────────────────────────────────────────────────

def siglip_global_loss(img_feats, txt_feats, logit_scale, logit_bias):
    logits = logit_scale.exp() * (img_feats @ txt_feats.T) + logit_bias
    B      = logits.shape[0]
    labels = 2 * torch.eye(B, device=logits.device) - 1
    return -F.logsigmoid(labels * logits).mean()


def bbox_patch_mask(bbox, orig_size, n_side, eval_size):
    W, H = orig_size
    x1, y1, x2, y2 = bbox
    sx, sy = eval_size / W, eval_size / H
    x1, y1, x2, y2 = x1*sx, y1*sy, x2*sx, y2*sy
    patch_size = eval_size / n_side
    rows = torch.arange(n_side, dtype=torch.float32)
    cols = torch.arange(n_side, dtype=torch.float32)
    cy = (rows + 0.5) * patch_size
    cx = (cols + 0.5) * patch_size
    grid_cy, grid_cx = torch.meshgrid(cy, cx, indexing='ij')
    mask = (grid_cx >= x1) & (grid_cx <= x2) & (grid_cy >= y1) & (grid_cy <= y2)
    return mask.reshape(-1)


def filip_region_loss(patch_feats_n, phrase_token_feats, bbox_masks,
                      logit_scale, logit_bias):
    """
    Vectorised FILIP-style region-phrase contrastive loss.

    patch_feats_n    : (B, N, D) L2-normalised patch features
    phrase_token_feats: list of B tensors, each (T_i, D) L2-normalised token feats
    bbox_masks       : (B, N) bool — which patches belong to each item's bbox

    Replaces the O(B²) Python loop with a single (B·T_max, D) × (D, B·K_max)
    matmul, keeping peak memory ~49 MB at B=128, T=15, K=100, D=768.
    """
    B, N, D = patch_feats_n.shape
    device  = patch_feats_n.device
    dtype   = patch_feats_n.dtype

    # ── Pad phrase tokens → (B, T_max, D) ────────────────────────────────────
    T_max = max(t.shape[0] for t in phrase_token_feats)
    tokens_pad  = patch_feats_n.new_zeros(B, T_max, D)
    phrase_mask = torch.zeros(B, T_max, dtype=torch.bool, device=device)
    for i, t in enumerate(phrase_token_feats):
        tokens_pad[i, :t.shape[0]]  = t
        phrase_mask[i, :t.shape[0]] = True

    # ── Extract + pad bbox patches → (B, K_max, D) ───────────────────────────
    bbox_patches = [patch_feats_n[j][bbox_masks[j]] for j in range(B)]
    K_max = max(max(p.shape[0] for p in bbox_patches), 1)
    patches_pad = patch_feats_n.new_zeros(B, K_max, D)
    patch_mask  = torch.zeros(B, K_max, dtype=torch.bool, device=device)
    for j, p in enumerate(bbox_patches):
        if p.shape[0] > 0:
            patches_pad[j, :p.shape[0]] = p
            patch_mask[j, :p.shape[0]]  = True

    # ── One matmul → (B, T_max, B, K_max) ────────────────────────────────────
    sim = (tokens_pad.reshape(B * T_max, D) @
           patches_pad.reshape(B * K_max, D).T
           ).reshape(B, T_max, B, K_max)

    # mask padding patches with -inf before max
    sim = sim.masked_fill(~patch_mask[None, None], float('-inf'))

    # max over patches → (B, T_max, B); zero out empty bboxes
    max_sim   = sim.amax(dim=-1)
    empty_box = ~patch_mask.any(dim=-1)                          # (B,)
    max_sim   = max_sim.masked_fill(empty_box[None, None, :], 0.0)

    # zero out padding tokens; mean over valid tokens → (B, B) scores
    max_sim = max_sim.masked_fill(~phrase_mask[:, :, None], 0.0)
    n_toks  = phrase_mask.sum(dim=1).clamp(min=1).to(dtype)      # (B,)
    scores  = max_sim.sum(dim=1) / n_toks[:, None]               # (B, B)

    logits = logit_scale.exp() * scores + logit_bias
    labels = 2 * torch.eye(B, device=device) - 1
    return -F.logsigmoid(labels * logits).mean()


print('Loss functions defined: siglip_global_loss, filip_region_loss')

Loss functions defined: siglip_global_loss, filip_region_loss


In [9]:
# ── 7. Feature extraction helpers ─────────────────────────────────────────────
import types

def get_image_feats(model, pixel_values):
    """Global image embedding via MHAP pooler_output, L2-normalised.
    Uses pooler_output so the global loss gradient does not collapse patch tokens.
    """
    out = model.vision_model(pixel_values=pixel_values)
    return F.normalize(out.pooler_output, dim=-1)   # (B, D)


def get_patch_feats(model, pixel_values):
    """Per-patch features after post_layernorm, L2-normalised. Shape (B, N, D)."""
    out = model.vision_model(pixel_values=pixel_values)
    return F.normalize(out.last_hidden_state, dim=-1)   # (B, N, D)


def get_text_feats(model, input_ids, attention_mask=None):
    """Global text embedding: EOS token, L2-normalised. Shape (B, D)."""
    out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return F.normalize(out.last_hidden_state[:, -1, :], dim=-1)   # (B, D)


def get_phrase_token_feats(model, processor, phrases, device):
    """Per-token text features for a list of phrases. Returns list of (T_i, D) tensors."""
    inputs = processor(text=phrases, return_tensors='pt',
                       padding=True, truncation=True).to(device)
    out    = model.text_model(**inputs)
    hs     = out.last_hidden_state   # (B, T, D)
    result = []
    for i, phrase in enumerate(phrases):
        if 'attention_mask' in inputs:
            length = inputs['attention_mask'][i].sum().item()
        else:
            length = hs.shape[1]
        tok_feats = F.normalize(hs[i, :length, :], dim=-1)  # (T, D)
        result.append(tok_feats)
    return result


print('Feature extraction helpers defined.')

Feature extraction helpers defined.


In [10]:
# ── 8. Quick VOC eval ─────────────────────────────────────────────────────────
# Reuses the same eval pipeline as 02_b0_benchmark (MaskCLIP-style extraction).

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

VOC_ROOT = Path('/tmp/voc')
VOC_ROOT.mkdir(exist_ok=True)
voc_val  = tvd.VOCSegmentation(root=str(VOC_ROOT), year='2012',
                                image_set='val', download=True)
print(f'VOC 2012 val: {len(voc_val)} images')


def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    return self.out_proj(self.v_proj(hidden_states)), None


def quick_voc_eval(model, processor, n_images=200, tau_seg=0.0, device=DEVICE):
    """Run MaskCLIP-style zero-shot segmentation on n_images VOC val images."""
    model.eval()
    n_fg   = len(VOC_CLASSES)
    n_cls  = n_fg + 1

    # Encode text classes once
    prompts = [f'a photo of a {c}' for c in VOC_CLASSES]
    txt_in  = processor(text=prompts, return_tensors='pt',
                        padding=True).to(device)
    with torch.no_grad():
        txt_feats = F.normalize(
            model.text_model(**txt_in).last_hidden_state[:, -1, :], dim=-1
        )   # (20, D)

    # Patch extraction with MaskCLIP-style last attention bypass.
    # Patching self_attn.forward means model.vision_model(...) will use the bypass
    # and still apply post_layernorm, so last_hidden_state is correctly normalised.
    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    tp = torch.zeros(n_cls, dtype=torch.float64)
    fp = torch.zeros(n_cls, dtype=torch.float64)
    fn = torch.zeros(n_cls, dtype=torch.float64)
    gt_present = torch.zeros(n_cls, dtype=torch.bool)

    try:
        for i in tqdm(range(n_images), desc='VOC eval', leave=False):
            pil_img, target = voc_val[i]
            orig_w, orig_h  = pil_img.size
            gt = torch.from_numpy(np.array(target)).long()

            pix = processor(images=pil_img, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)

            # last_hidden_state is after post_layernorm — matches text feature space
            patch_n = F.normalize(out.last_hidden_state[0], dim=-1)   # (N, D)
            sim     = patch_n @ txt_feats.T                            # (N, 20)
            n_side  = int(sim.shape[0] ** 0.5)
            sim_up  = F.interpolate(
                sim.reshape(n_side, n_side, n_fg).permute(2,0,1).unsqueeze(0).float(),
                size=(orig_h, orig_w), mode='bilinear', align_corners=False
            ).squeeze(0).permute(1, 2, 0)

            max_sim, pred = sim_up.max(dim=-1)
            pred = pred + 1
            pred[max_sim < tau_seg] = 0
            pred = pred.cpu()

            valid = (gt != 255)
            pv, gv = pred[valid], gt[valid]
            for c in range(n_cls):
                pc = (pv == c); gc = (gv == c)
                tp[c] += (pc & gc).sum()
                fp[c] += (pc & ~gc).sum()
                fn[c] += (~pc & gc).sum()
                if gc.any(): gt_present[c] = True
    finally:
        last_attn.forward = orig_fwd

    iou   = tp / (tp + fp + fn).clamp(min=1e-6)
    miou  = iou[gt_present].mean().item() * 100
    return miou


print('quick_voc_eval defined.')

100%|██████████| 2.00G/2.00G [01:33<00:00, 21.4MB/s] 


VOC 2012 val: 1449 images
quick_voc_eval defined.


In [11]:
# ── 9. Training loop ─────────────────────────────────────────────────────────

def train_one_epoch(model, processor, train_ds, lambda_region,
                    optimizer, scheduler, device, cfg,
                    run_tag='', step_offset=0, eval_every=100):
    model.train()
    loader = DataLoader(
        train_ds,
        batch_size   = cfg['batch_size'],
        shuffle      = True,
        num_workers  = 2,
        collate_fn   = FlickrSigLIPDataset.collate_fn,
        pin_memory   = device == 'cuda',
    )

    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

    total_loss = total_global = total_region = 0.0
    n_steps = 0

    pbar = tqdm(loader, desc=f'train {run_tag}')
    for pils, captions, orig_sizes, phrase_boxes_batch in pbar:

        # ── L_global: image ↔ caption ────────────────────────────────────────
        img_inputs = processor(images=pils, return_tensors='pt',
                               padding=True).to(device)
        txt_inputs = processor(text=captions, return_tensors='pt',
                               padding=True, truncation=True).to(device)

        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            img_feats = get_image_feats(model, img_inputs['pixel_values'])
            txt_feats = get_text_feats(model, txt_inputs['input_ids'])
            l_global  = siglip_global_loss(
                img_feats, txt_feats,
                model.logit_scale, model.logit_bias
            )

        # ── L_region: phrase ↔ bbox patches ──────────────────────────────────
        l_region = torch.tensor(0.0, device=device)

        if lambda_region > 0:
            triples = []
            for img_idx, pb_list in enumerate(phrase_boxes_batch):
                for phrase, bbox in pb_list:
                    triples.append((img_idx, phrase, bbox))

            if len(triples) >= 2:
                random.shuffle(triples)
                triples = triples[:cfg['region_batch']]

                reg_img_idxs = [t[0] for t in triples]
                reg_phrases  = [t[1] for t in triples]
                reg_bboxes   = [t[2] for t in triples]
                reg_orig_sz  = [orig_sizes[i] for i in reg_img_idxs]
                reg_pils     = [pils[i] for i in reg_img_idxs]
                reg_pix      = processor(images=reg_pils, return_tensors='pt',
                                         padding=True).pixel_values.to(device)

                with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
                    patch_feats_n    = get_patch_feats(model, reg_pix)
                    phrase_tok_feats = get_phrase_token_feats(
                        model, processor, reg_phrases, device
                    )
                    bbox_masks = torch.stack([
                        bbox_patch_mask(bbox, sz, cfg['n_side'], cfg['eval_size'])
                        for bbox, sz in zip(reg_bboxes, reg_orig_sz)
                    ]).to(device)
                    l_region = filip_region_loss(
                        patch_feats_n, phrase_tok_feats, bbox_masks,
                        model.logit_scale, model.logit_bias
                    )

        loss = l_global + lambda_region * l_region

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss   += loss.item()
        total_global += l_global.item()
        total_region += l_region.item() if isinstance(l_region, torch.Tensor) else 0.0
        n_steps      += 1
        global_step   = step_offset + n_steps

        wandb.log({
            f'{run_tag}/loss':         loss.item(),
            f'{run_tag}/loss_global':  l_global.item(),
            f'{run_tag}/loss_region':  l_region.item() if isinstance(l_region, torch.Tensor) else 0.0,
            f'{run_tag}/lr':           scheduler.get_last_lr()[0],
            f'{run_tag}/logit_scale':  model.logit_scale.item(),
            f'{run_tag}/logit_bias':   model.logit_bias.item(),
        }, step=global_step)

        pbar.set_postfix({
            'loss':   f'{total_loss/n_steps:.3f}',
            'global': f'{total_global/n_steps:.3f}',
            'region': f'{total_region/n_steps:.3f}',
        })

        # ── Periodic VOC eval ─────────────────────────────────────────────────
        if eval_every > 0 and n_steps % eval_every == 0:
            step_miou = quick_voc_eval(
                model, processor,
                n_images = cfg['eval_images'],
                tau_seg  = cfg['tau_seg'],
                device   = device,
            )
            wandb.log({f'{run_tag}/miou': step_miou}, step=global_step)
            pbar.write(f'  step {global_step:4d}  VOC mIoU = {step_miou:.2f}%')
            model.train()   # quick_voc_eval calls model.eval(); restore train mode

    stats = {
        'loss':   total_loss   / n_steps,
        'global': total_global / n_steps,
        'region': total_region / n_steps,
    }
    return stats, n_steps


def make_optimizer_scheduler(model, n_steps, cfg):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )
    def lr_lambda(step):
        if step < cfg['warmup_steps']:
            return step / max(1, cfg['warmup_steps'])
        progress = (step - cfg['warmup_steps']) / max(1, n_steps - cfg['warmup_steps'])
        return max(0.0, 0.5 * (1 + np.cos(np.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


print('Training loop defined.')

Training loop defined.


In [ ]:
# ── 10. Train M_human ────────────────────────────────────────────────────────

# Free any leftover model from a previous (possibly OOM'd) run
import gc
for _var in ['model_mh', 'opt_mh', 'sch_mh']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

base_model_mh = SiglipModel.from_pretrained(CFG['model_id'],
                                             token=os.environ['HF_TOKEN'])
model_mh = get_peft_model(base_model_mh, lora_cfg).to(DEVICE)
for n, p in model_mh.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

# Gradient checkpointing: recompute activations during backward instead of
# storing them. Cuts activation memory ~4x; costs ~33% extra compute per step.
# Required with PEFT: enable_input_require_grads() ensures frozen-param inputs
# still flow through the checkpointing mechanism.
model_mh.enable_input_require_grads()
model_mh.gradient_checkpointing_enable()

n_steps_mh     = len(train_ds) // CFG['batch_size'] * CFG['epochs']
opt_mh, sch_mh = make_optimizer_scheduler(model_mh, n_steps_mh, CFG)

print(f'=== Training M_human (λ_region={CFG["lambda_region"]}) ===')
stats_mh, mh_steps = train_one_epoch(
    model_mh, processor, train_ds,
    lambda_region = CFG['lambda_region'],
    optimizer     = opt_mh,
    scheduler     = sch_mh,
    device        = DEVICE,
    cfg           = CFG,
    run_tag       = 'mhuman',
    step_offset   = 0,
)
print(f'Train stats: {stats_mh}')

model_mh.save_pretrained(f'{CFG["ckpt_dir"]}/mhuman_lam{CFG["lambda_region"]}_epoch1')
print('Checkpoint saved.')

mh_miou = quick_voc_eval(model_mh, processor,
                          n_images=CFG['eval_images'], tau_seg=CFG['tau_seg'])
print(f'\nM_human VOC mIoU ({CFG["eval_images"]} images): {mh_miou:.2f}%')
wandb.log({
    'eval/mhuman_miou':          mh_miou,
    'mhuman/epoch_loss':         stats_mh['loss'],
    'mhuman/epoch_loss_global':  stats_mh['global'],
    'mhuman/epoch_loss_region':  stats_mh['region'],
}, step=mh_steps)

GPU free: 22.6 GB


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

=== Training M_human (λ_region=0.5) ===


train mhuman:   0%|          | 0/907 [00:00<?, ?it/s]

VOC eval:   0%|          | 0/200 [00:00<?, ?it/s]

...

  step  400  VOC mIoU = 0.36%


VOC eval:   0%|          | 0/200 [00:00<?, ?it/s]

  step  500  VOC mIoU = 0.11%


VOC eval:   0%|          | 0/200 [00:00<?, ?it/s]

  step  700  VOC mIoU = 0.08%


In [13]:
# ── 11. Results ───────────────────────────────────────────────────────────────
print('=' * 50)
print(f'  VOC 2012 val mIoU  ({CFG["eval_images"]} images, MaskCLIP-style)')
print('=' * 50)
print(f'  B0  (frozen, report.md) : {B0_MIOU:6.2f}%')
print(f'  M_human (λ={CFG["lambda_region"]})        : {mh_miou:6.2f}%')
print()
print(f'  M_human − B0            : {mh_miou - B0_MIOU:+.2f}%')
print('=' * 50)

print('\nTraining loss summary:')
print(f'  M_human global loss : {stats_mh["global"]:.3f}')
print(f'  M_human region loss : {stats_mh["region"]:.3f}')

wandb.summary.update({
    'b0_miou':            B0_MIOU,
    'mhuman_miou':        mh_miou,
    'delta_mhuman_vs_b0': mh_miou - B0_MIOU,
})
wandb.finish()
print('W&B run finished.')

: 